<a href="https://colab.research.google.com/github/tranlinh102/BERT_Vietnamse_Sentiment_Assistant/blob/main/BERT_Vietnamse_Sentiment_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install streamlit pyngrok transformers torch -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 44.1 MB/s eta 0:00:00


In [ ]:
%%writefile app.py
import streamlit as st
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import sqlite3, datetime, os
import re

# Xử lý biến thể tiếng Việt

VIET_VARIANTS = {
    "rat": "rất",
    "ratt": "rất",
    "dc": "được",
    "đc": "được",
    "dcj": "được",
    "hok": "không",
    "hum": "hôm",
    "hom": "hôm",
    "vs": "với",
    "k": "không",
    "ko": "không",
    "khong": "không",
    "mn": "mọi người",
    "ad": "admin",
}

def normalize_vietnamese(text: str):
    text = text.lower().strip()

    # thay biến thể bằng từ chuẩn
    for k, v in VIET_VARIANTS.items():
        pattern = r"\b" + re.escape(k) + r"\b"
        text = re.sub(pattern, v, text)

    # xoá ký tự linh tinh (giữ dấu tiếng Việt)
    text = re.sub(r"[^a-zA-Z0-9À-ỹ\s,.!?\-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    # giới hạn 50 ký tự
    if len(text) > 50:
        text = text[:50].rstrip()

    return text


# Load Model (cache)

model_name = "Zonecb/my-phobert-sentiment-v2"

@st.cache_resource
def load_model():
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    clf = pipeline("text-classification", model=model, tokenizer=tokenizer)
    return clf

def get_classifier():
    return load_model()

classifier = get_classifier()


# Map nhãn
LABEL_MAP = {
    "positive": "POSITIVE",
    "negative": "NEGATIVE",
    "neutral":  "NEUTRAL"
}


# SQLite
def init_db():
    conn = sqlite3.connect("sentiments.db")
    cursor = conn.cursor()
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS sentiments (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        text TEXT,
        sentiment TEXT,
        timestamp TEXT
    )
    """)
    conn.commit()
    return conn, cursor

conn, cursor = init_db()

def save_to_db(text, sentiment):
    ts = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    cursor.execute("INSERT INTO sentiments(text, sentiment, timestamp) VALUES (?, ?, ?)",
                   (text, sentiment, ts))
    conn.commit()

def load_latest():
    cursor.execute("SELECT * FROM sentiments ORDER BY timestamp DESC LIMIT 50")
    return cursor.fetchall()


# UI
st.title("Phân Tích Cảm Xúc Văn Bản — PhoBERT")

user_text = st.text_area("Nhập câu cần phân tích cảm xúc:", height=150, key="input_text")

if st.button("Phân tích"):
    if len(user_text.strip()) < 5:
        st.warning("Văn bản quá ngắn.")
    else:
        # TIỀN XỬ LÝ TIẾNG VIỆT
        clean_text = normalize_vietnamese(user_text)

        # chạy model
        result = classifier(clean_text)[0]
        raw_label = result["label"].lower()
        score = result["score"]

        # rule: score < 0.5 → NEUTRAL
        if score < 0.5:
            sentiment = "NEUTRAL"
        else:
            sentiment = LABEL_MAP.get(raw_label, "NEUTRAL")

        save_to_db(clean_text, sentiment)

        # Display result with icon and color
        if sentiment == "POSITIVE":
            st.success(f"Kết quả: **Tích cực** (score={score:.4f})")
        elif sentiment == "NEGATIVE":
            st.error(f"Kết quả: **Tiêu cực** (score={score:.4f})")
        else: # NEUTRAL
            st.info(f"Kết quả: **Trung tính** (score={score:.4f})")

st.subheader("Lịch sử dự đoán gần đây")

rows = load_latest()
for r in rows:
    st.write(f"- `{r[3]}` | **{r[2]}** → {r[1]}")

if st.button("Xoá toàn bộ dữ liệu"):
    if os.path.exists("sentiments.db"):
        os.remove("sentiments.db")
        st.success("Đã xoá toàn bộ dữ liệu! Vui lòng reload ứng dụng.")
        st.stop()


Overwriting app.py


In [ ]:
# Chạy Streamlit + ngrok trong Colab
from pyngrok import ngrok
import os
import time

# Set token ngrok
ngrok.set_auth_token("35hwjW82MBTs4MgHefNQVyfQDfw_4Tuwk82VXZv59obFo2Dd3")

# Mở tunnel trước
public_url = ngrok.connect(8501)
print("Ngrok URL:", public_url)

# Chạy Streamlit trong background
os.system("streamlit run app.py --server.port 8501 &")

# Chờ một chút để server khởi động
time.sleep(5)

print(f"Ứng dụng đang chạy: {public_url}")


Ngrok URL: NgrokTunnel: "https://cirrose-theron-unworkmanlike.ngrok-free.dev" -> "http://localhost:8501"
Ứng dụng đang chạy: NgrokTunnel: "https://cirrose-theron-unworkmanlike.ngrok-free.dev" -> "http://localhost:8501"


In [ ]:
# Ngắt kết nối tất cả các tunnel ngrok đang hoạt động
ngrok.kill()

print("Đã ngắt kết nối tất cả các tunnel ngrok.")